# SpaceX Falcon 9 — Web Scraping Launch Records from Wikipedia

**Objective:** scrape the Falcon 9 launch records HTML tables from the Wikipedia page *List of Falcon 9 and Falcon Heavy launches* (snapshot of 9 June 2021), parse them with BeautifulSoup and convert them into a pandas dataframe.

> Runs in **JupyterLite**. The page is requested through the Wikipedia API (`origin=*` allows browser requests).

In [ ]:
import piplite
await piplite.install(['numpy', 'pandas', 'beautifulsoup4'])

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from bs4 import BeautifulSoup

In [ ]:
from js import fetch
import io, json

async def get_text(url):
    resp = await fetch(url)
    if not resp.ok:
        raise Exception(f"HTTP {resp.status} for {url}")
    return await resp.text()

async def get_json(url):
    return json.loads(await get_text(url))

async def get_csv(url):
    return pd.read_csv(io.StringIO(await get_text(url)))

## Helper functions to process the HTML table cells

In [ ]:
def date_time(table_cells):
    """Returns the date and time from the HTML table cell"""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """Returns the booster version from the HTML table cell"""
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """Returns the landing status from the HTML table cell"""
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    """Returns the column name from the HTML table header cell"""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colunm_name = ' '.join(row.contents)
    # Filter the digit and empty names
    if not (colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name

## Task 1: Request the Falcon 9 launch Wiki page

In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# Same snapshot (oldid=1027686922) requested through the Wikipedia API, which allows browser (CORS) requests
api_url = ("https://en.wikipedia.org/w/api.php?action=parse&oldid=1027686922"
           "&prop=text&format=json&formatversion=2&origin=*")
page = await get_json(api_url)
html = page['parse']['text']
print("HTML characters downloaded:", len(html))

In [ ]:
soup = BeautifulSoup(html, 'html.parser')
print(page['parse']['title'])

## Task 2: Extract all column names from the HTML table header

In [ ]:
html_tables = soup.find_all('table', "wikitable plainrowheaders collapsible")
print("Launch tables found:", len(html_tables))

first_launch_table = html_tables[0]
print(first_launch_table.prettify()[:1500])

In [ ]:
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

## Task 3: Create a dataframe by parsing the launch HTML tables

In [ ]:
launch_dict = dict.fromkeys(column_names)

# Remove an irrelevant column
launch_dict.pop('Date and time ( )', None)

# Initialize launch_dict with each value as an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

In [ ]:
extracted_row = 0
# Extract each table
for table_number, table in enumerate(html_tables):
    # get table row
    for rows in table.find_all("tr"):
        # check to see if first table heading is a number corresponding to a launch number
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
        # get table element
        row = rows.find_all('td')
        # if it is a number save cells in a dictionary
        if flag:
            extracted_row += 1
            # Flight Number value
            launch_dict['Flight No.'].append(flight_number)

            # Date and time value
            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            time = datatimelist[1]
            launch_dict['Time'].append(time)

            # Booster version
            bv = booster_version(row[1])
            if not (bv):
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)

            # Launch site
            launch_site = row[2].a.string
            launch_dict['Launch site'].append(launch_site)

            # Payload
            payload = row[3].a.string
            launch_dict['Payload'].append(payload)

            # Payload mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            # Orbit
            orbit = row[5].a.string
            launch_dict['Orbit'].append(orbit)

            # Customer
            customer = row[6].a.string if row[6].a else row[6].text.strip()
            launch_dict['Customer'].append(customer)

            # Launch outcome
            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)

            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print("Rows extracted:", extracted_row)

In [ ]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
print(df.shape)
df.head()

In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)

## Author

Jhoselyn Vera